<a href="https://colab.research.google.com/github/lcbjrrr/genai/blob/main/Google_ADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ADK offers several key advantages for developers building agentic applications:
*texto em itálico*
*   **Multi-Agent System Design**: Easily build applications composed of multiple, specialized agents arranged hierarchically. Agents can coordinate complex tasks, delegate sub-tasks using LLM-driven transfer or explicit AgentTool invocation, enabling modular and scalable solutions.
*   **Rich Tool Ecosystem**: Equip agents with diverse capabilities. ADK supports integrating custom functions (FunctionTool), using other agents as tools (AgentTool), leveraging built-in functionalities like code execution, and interacting with external data sources and APIs (e.g., Search, Databases). Support for long-running tools allows handling asynchronous operations effectively.
*   **Flexible Orchestration**: Define complex agent workflows using built-in workflow agents (SequentialAgent, ParallelAgent, LoopAgent) alongside LLM-driven dynamic routing. This allows for both predictable pipelines and adaptive agent behavior.
*   **Integrated Developer Tooling**: Develop and iterate locally with ease. ADK includes tools like a command-line interface (CLI) and a Developer UI for running agents, inspecting execution steps (events, state changes), debugging interactions, and visualizing agent definitions.
*   **Native Streaming Support**: Build real-time, interactive experiences with ADK Gemini Live API Toolkit that provides native support for bidirectional streaming (text and audio). This integrates seamlessly with underlying capabilities like the Gemini Live API for the Gemini Developer API (or for Agent Platform), often enabled with simple configuration changes.
*   **Built-in Agent Evaluation**: Assess agent performance systematically. The framework includes tools to create multi-turn evaluation datasets and run evaluations locally (via CLI or the dev UI) to measure quality and guide improvements.
*   **Broad LLM Support**: While optimized for Google's Gemini models, the framework is designed for flexibility, allowing integration with various LLMs (potentially including open-source or fine-tuned models) through its BaseLlm interface.
*   **Artifact Management**: Enable agents to handle files and binary data. The framework provides mechanisms (ArtifactService, context methods) for agents to save, load, and manage versioned artifacts like images, documents, or generated reports during their execution.
*   **Extensibility and Interoperability**: ADK promotes an open ecosystem. While providing core tools, it allows developers to easily integrate and reuse third-party tools and data connectors.
*   **State and Memory Management**: Automatically handles short-term conversational memory (State within a Session) managed by the SessionService. Provides integration points for longer-term Memory services, allowing agents to recall user information across multiple sessions.

![](https://adk.dev/assets/adk-lifecycle.png)


### Agent Development Kit (ADK)

Build, Evaluate, and Deploy agents, seamlessly!

ADK is designed to empower developers to build, manage, evaluate, and deploy AI-powered agents. It provides a robust and flexible environment for creating both conversational and non-conversational agents, capable of handling complex tasks and workflows.

![](https://adk.dev/assets/adk-components.png)

### Core Concepts

ADK is built around a few key primitives and concepts that make it powerful and flexible. Here are the essentials:

*   **Agent**: The fundamental worker unit designed for specific tasks. Agents can use language models (LlmAgent) for complex reasoning, or act as deterministic controllers of the execution, which are called "workflow agents" (SequentialAgent, ParallelAgent, LoopAgent).
*   **Tool**: Gives agents abilities beyond conversation, letting them interact with external APIs, search information, run code, or call other services.
*   **Callbacks**: Custom code snippets you provide to run at specific points in the agent's process, allowing for checks, logging, or behavior modifications.
*   **Session Management (Session & State)**: Handles the context of a single conversation (Session), including its history (Events) and the agent's working memory for that conversation (State).
*   **Memory**: Enables agents to recall information about a user across multiple sessions, providing long-term context (distinct from short-term session State).
*   **Artifact Management (Artifact)**: Allows agents to save, load, and manage files or binary data (like images, PDFs) associated with a session or user.
*   **Code Execution**: The ability for agents (usually via Tools) to generate and execute code to perform complex calculations or actions.
*   **Planning**: An advanced capability where agents can break down complex goals into smaller steps and plan how to achieve them like a ReAct planner.
*   **Models**: The underlying LLM that powers LlmAgents, enabling their reasoning and language understanding abilities.
*   **Event**: The basic unit of communication representing things that happen during a session (user message, agent reply, tool use), forming the conversation history.
*   **Runner**: The engine that manages the execution flow, orchestrates agent interactions based on Events, and coordinates with backend services.







```
pip install google-adk
```



In [5]:
!pip install google-adk




```
export GEMINI_API_KEY="AQ."
```





```
adk create stocks_agent
```



In [9]:
%%writefile stocks_agent/agent.py
from __future__ import annotations
import random
from google.adk.agents import Agent

def get_quote_f(ticker: str) -> int:
    """Gets real-time stock quotes for supported tickers like AAPL or NVDA."""
    print("===> get_quote_f", ticker)
    if ticker == "AAPL":
        return random.randint(280, 310)
    elif ticker == "NVDA":
        return random.randint(190, 220)
    else:
        return 999

def buy_signal_f(ticker: str, quote: float) -> str:
    """Evaluates whether a given stock is currently a good buy based on its ticker symbol and price quote."""
    print("===> buy_signal_f", ticker, quote)

    buy = False
    if ticker == "AAPL" and quote < 300:
        buy = True
    elif ticker == "NVDA" and quote < 200:
        buy = True

    if buy:
        return f"This is a good moment to BUY {ticker} at ${quote}."
    else:
        return f"This is NOT a moment to invest in {ticker}. It is too expensive at ${quote}."

root_agent = Agent(
    name="my_first_agent",
    model="gemini-2.5-flash",
    description="Assistant for stock prices and buy analysis",
    instruction=( '''
        You are a financial assistant. "
        When asked if a stock is a good buy, first retrieve the current price using get_quote_f, "
        and then pass that price to buy_signal_f to evaluate the buy signal."
        You CAN perform basic arithmetic and remember previous stock quotes mentioned earlier in this chat. "
    '''),
    tools=[get_quote_f, buy_signal_f]
)

# What is current Apple stock quote? Should I buy it now?
# What is its the most up to date Apple quote ? And what was the last quote? What is the percentage variation between them 100*(current-last)/last ?

Overwriting stocks_agent/agent.py




```
adk run stocks_agent
```
or `adk web stocks_agent`



In [10]:
!adk run stocks_agent


Log setup complete: /tmp/agents_log/agent.20260830_160453.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.13/dist-packages/google/adk/cli/cli.py:335: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.13/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent my_first_agent, type exit to exit.
[user]: # What is current Apple stock quote? Should I buy it now? 
/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureNa

### Final version

In [31]:
%%writefile stocks_agent/agent.py
from __future__ import annotations
import random
from google.adk.agents import Agent
from google.adk.tools.tool_context import ToolContext

def get_quote_f(ticker: str) -> int:
    """Gets real-time stock quotes for supported tickers (AAPL, NVDA)."""
    print(f"===> get_quote_f: {ticker}")
    if ticker == "AAPL":
        return random.randint(280, 310)
    elif ticker == "NVDA":
        return random.randint(190, 220)
    else:
        return 999

def buy_stock_f(ticker: str, tool_context: ToolContext) -> str:
    """Evaluates whether a stock is a good buy based on market price and executes the trade if cash is available."""
    print(f"===> buy_stock_f: {ticker}")

    # 1. Fetch live quote
    quote = get_quote_f(ticker)

    # 2. Check Buy Signal criteria (AAPL < $300, NVDA < $200)
    is_good_buy = (ticker == "AAPL" and quote < 300) or (ticker == "NVDA" and quote < 200)

    if not is_good_buy:
        return f"BUY SIGNAL FAILED: Current price for {ticker} is ${quote}, which exceeds target purchase threshold."

    # 3. Access portfolio state
    qtys=random.randint(1,10)
    state = tool_context.state
    portfolio = state.get("portfolio", {"CASH": 9999.0})
    price = quote * qtys

    # 4. Check funds and execute transaction
    if portfolio["CASH"] >= price:
        portfolio["CASH"] -= price
        portfolio[ticker] = portfolio.get(ticker, 0) + qtys
        state["portfolio"] = portfolio
        return (
            f"BUY SIGNAL CONFIRMED & EXECUTED: Bought {qtys} shares of {ticker} at ${quote}/share "
            f"(${price:.2f} total). Remaining CASH: ${portfolio['CASH']:.2f}"
        )
    else:
        return (
            f"BUY SIGNAL CONFIRMED BUT FAILED ON FUNDS: Target price is ${quote} (${price:.2f} total), "
            f"but available CASH is ${portfolio['CASH']:.2f}."
        )

def get_portfolio_total(tool_context: ToolContext) -> str:
    """Calculates and returns current holdings, cash balance, and total valuation."""
    portfolio = tool_context.state.get("portfolio", {"CASH": 9999.0})
    total = portfolio["CASH"]
    holdings = []

    for ticker, qty in portfolio.items():
        if ticker != "CASH":
            val = qty * get_quote_f(ticker)
            total += val
            holdings.append(f"{ticker}: {qty} shares")

    return f"Portfolio CASH: ${portfolio['CASH']:.2f} | Holdings: {holdings} | Total Valuation: ${total:.2f}"

root_agent = Agent(
    name="portfolio_agent",
    model="gemini-2.5-flash",
    instruction=('''
        You are an automated stockbroker.
        When asked to buy stock, call buy_stock_f. It automatically checks market price,
        evaluates the buy signal, and executes the trade if funds are sufficient.
        Use get_portfolio_total whenever requested to report portfolio valuation.
        You CAN perform basic arithmetic and remember previous stock quotes mentioned earlier in this chat.
        By the end of each interaction the portfolio should be printed out, using the get_portfolio_total tool.
    '''),tools=[get_quote_f, buy_stock_f, get_portfolio_total]
)

# What is current Apple stock quote? Should I buy it now?
# What is its the most up to date Apple quote ? And what was the last quote? What is the percentage variation between them 100*(current-last)/last ?

Overwriting stocks_agent/agent.py


In [32]:
!adk run stocks_agent

Log setup complete: /tmp/agents_log/agent.20260830_163945.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.13/dist-packages/google/adk/cli/cli.py:335: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.13/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent portfolio_agent, type exit to exit.
[user]: # What is current Apple stock quote? Should I buy it now? 
/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureN